# Kv2.1 MolProbity / clashscore analysis

All six WT/L403A/F412L × vanilla/masked ensembles are compared. The conformational extension
uses the exact structures and existing chain-resolved coordinates from
`Kv21_L403A_conformational_validation.ipynb`. Established tetramer/interface/alignment QC is retained.

## Reproducible setup
Run from the repository root or channel directory. Dependencies: NumPy, pandas, SciPy, statsmodels, Matplotlib, seaborn, and Jupyter. MolProbity tables remain in the local downloaded directory. No upload or move is performed. Shared functions live in `shared/molprobity_analysis.py`, as in the existing analyses.

In [1]:
from pathlib import Path
import sys, os, json, platform, importlib.metadata
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data_access.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from shared import molprobity_analysis as mp
CHANNEL = 'kv21'
OUT = mp.output_dir(CHANNEL)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 24)
mp.apply_kv21_style()


## Data-access boundary
Set `VGIC_MOLPROBITY_ROOT` or `DATA_ROOT` to the downloaded directory. Future Hugging Face migration only needs to replace the loader with explicit paths through the existing `shared.data_access.resolve`; downstream analysis accepts the same DataFrame. Existing structural-QC and conformational inputs already use that resolver, including local/cache/offline behavior.

In [2]:
DATA_ROOT = os.environ.get('VGIC_MOLPROBITY_ROOT')
raw, inventory, total_csv_files = mp.load_molprobity(CHANNEL, DATA_ROOT)
inventory.to_csv(OUT/'tables/input_inventory.csv', index=False)
print(f'{total_csv_files} CSVs in downloaded directory; {len(inventory)} for this channel; {len(raw):,} rows')
display(inventory)


12 CSVs in downloaded directory; 6 for this channel; 35,994 rows


,csv_file,rows,sha256
0,Kv21_f412l_masked_v3_50pct_clashscores.csv,6000,5f37cebcd58e1713a5d4db864a6bf465fdbb7e7ef5a3a3...
1,Kv21_f412l_vanilla_clashscores.csv,6000,6805e145186fa2566a9151b437e1d006f4c7d4931988de...
2,Kv21_l403a_masked_v3_50pct_clashscores.csv,6000,752d77878503e66d223d4452be5e8d0d7e1b5424363211...
3,Kv21_l403a_vanilla_clashscores.csv,5994,021390d94f175d45838b4e9dede98a20b71808c77e39b2...
4,Kv21_wt_masked_v3_50pct_clashscores.csv,6000,a48a3f5f43c65c0660543d357529df6e7e7134e4aec00d...
5,Kv21_wt_vanilla_clashscores.csv,6000,ad9c53ab87b8228adc075727b07ce483e87cb7f56a5b3a...


## Audit before plotting
The expected grid uses each ensemble’s recorded seed labels, AF models 1–5, r0–r10, and final. Missing whole trajectories/seeds are additionally checked against the existing model manifest. Raw input hashes are saved above. NaN recycle is expected only for final/base models; missing rank is allowed and reported.

In [3]:
data, audit = mp.audit_data(raw, CHANNEL, OUT)
display(audit)
display(pd.read_csv(OUT/'tables/missing_snapshots.csv'))
display(pd.read_csv(OUT/'tables/manifest_discrepancies.csv'))
display(pd.read_csv(OUT/'tables/row_anomalies.csv'))
cohorts = mp.select_cohorts(data, CHANNEL, OUT)
summary = pd.concat([mp.describe(frame, name) for name, frame in cohorts.items()], ignore_index=True)
summary.to_csv(OUT/'tables/descriptive_statistics.csv', index=False)
display(summary[(summary.cohort=='all_recorded') & summary.metric.eq('clashscore')].round(3))
display(pd.read_csv(OUT/'tables/selection_audit.csv'))


Audit: 35,994 rows, 0 failed, 0 invalid metric rows; 6 absent expected snapshots; 0 explicitly excluded rows.


,ensemble,rows,ok,ok_fraction,eligible,failed,missing_clashscore,nonfinite_clashscore,duplicate_names,duplicate_keys,missing_grid_snapshots,seeds,seed_min,seed_max,missing_rank,final_r10_pairs,final_r10_identical_metrics,manifest_rows,manifest_missing,manifest_extra
0,F412L | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
1,F412L | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,78,177,0,500,500,6000,0,0
2,L403A | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
3,L403A | vanilla,5994,5994,1.0,5994,0,0,0,0,0,6,100,55,154,5994,499,499,5994,0,0
4,WT | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
5,WT | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,48,147,0,500,500,6000,0,0


,ensemble,seed,af_model,snapshot
0,L403A | vanilla,154,5,final
1,L403A | vanilla,154,5,r10
2,L403A | vanilla,154,5,r6
3,L403A | vanilla,154,5,r7
4,L403A | vanilla,154,5,r8
5,L403A | vanilla,154,5,r9


,ensemble,model_name,issue


,run_name,input_folder,model_name,rank,af_model,seed,recycle,snapshot,clashscore,n_bad_clashes,max_overlap,status,...,source_path,genotype,protocol,ensemble,csv_file,analysis_eligible,malformed_numeric,invalid_identity,duplicate_name,duplicate_key,invalid_metric,failed_status_or_exit


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
0,all_recorded,F412L | masked v3,clashscore,6000,100,47.200,41.738,56.422,14.685,51.069,16.012,25.24,204.50
3,all_recorded,F412L | vanilla,clashscore,6000,100,45.810,36.228,55.920,19.693,48.977,16.940,22.60,218.63
6,all_recorded,L403A | masked v3,clashscore,6000,100,43.455,37.160,49.740,12.580,45.737,13.285,25.33,157.83
9,all_recorded,L403A | vanilla,clashscore,5994,100,45.785,38.585,57.598,19.013,50.238,17.231,22.65,203.36
12,all_recorded,WT | masked v3,clashscore,6000,100,42.450,37.260,49.000,11.740,45.742,14.342,24.53,239.61
15,all_recorded,WT | vanilla,clashscore,6000,100,44.640,38.400,56.415,18.015,49.229,16.020,24.45,143.01


,ensemble,selection,source,requested,matched,excluded_failed,excluded_by_structural_qc
0,F412L | masked v3,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-12_Kv2.1_f412l_masked...,3643,3643,0,2357
1,F412L | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_f412l_vanill...,4521,4521,0,1479
2,L403A | masked v3,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-12_Kv2.1_l403a_masked...,4599,4599,0,1401
3,L403A | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_l403a_vanill...,4403,4403,0,1591
4,WT | masked v3,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-12_Kv2.1_wt_masked_v3...,4366,4366,0,1634
5,WT | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF...,4591,4591,0,1409


## Statistical design and interpretation

The complete downloaded table is audited before filtering. `status != ok`, nonzero exit codes,
nonfinite/negative diagnostics and malformed identities are reported explicitly. Ambiguous identities
or mismatched manifests stop execution. Empty anomaly CSVs still retain their headers.

Every ensemble has up to 100 seeds × 5 AF models × 11 recycle snapshots plus a final/base file.
The final/base file has identical recorded MolProbity metrics to r10 for every available trajectory;
that is **metric equality, not a coordinate checksum**. To avoid double weighting those observations,
`all_recycles` excludes final copies. `final_only` is a separate sensitivity analysis.

The primary scientific comparison uses the repository's existing QC subset, independently of clashscore:
3 Å convergence for Cav1.2/Nav1.5 and the established structural/interface/alignment-QC distance
selection for Kv2.1. All raw distributions and excluded counts remain visible. QC-selected results
describe survivors, not all generated models. No new clashscore cutoff is used.

For each metric, average snapshots within seed/AF-model trajectory, then average AF models within seed.
This gives each retained seed equal weight and avoids treating recycles or the five AF models as
independent replicates. Point estimates are seed-balanced mean differences B−A; 95% percentile
bootstrap intervals use 3,000 seed draws. When seed-label sets match completely, nominal paired-seed
t tests and the existing `paired_common_seed_bootstrap` are used. Otherwise Welch tests compare all
retained seed summaries. Both independent-seed and common-label paired sensitivities are exported.
Shared numeric labels do **not** establish shared random-number streams; no seeds are renumbered,
and rank is never a pairing key. Common-label sensitivity may change the population when ranges differ.

BH correction covers all planned pairwise comparisons × all three metrics within each channel/cohort;
primary, independent, and common-label p-values form separate families. Supplemental cohort families
are exploratory. The seed rank-biserial effect is P(B>A)−P(B<A) across seed summaries, with ties zero;
it describes marginal ordering even for nominally paired tests. Model medians/IQRs are descriptive
snapshot summaries and are a different estimand from the seed-balanced mean contrast.

AF-model/snapshot-adjusted regressions provide a sensitivity check with standard errors clustered by
recorded seed. Rank is an output ordering variable, is absent for Kv2.1 L403A vanilla, and is inspected
descriptively rather than controlled as if it were a randomized covariate. These uncertainty estimates
describe computational sampling, not biological replication or a validated clinical effect.

Clashscore is a normalized global clash statistic ([Phenix glossary](https://www.phenix-online.org/version_docs/1.8.2-1296/glossary.htm)).
`n_bad_clashes` is an absolute count and `max_overlap` is a single extreme; they are complementary
diagnostics, not interchangeable biological measures. The local export does not record Phenix version,
hydrogen options, the atom denominator, or atom-level clash identities. Absolute validation grades,
cross-channel biological comparisons, and residue-specific attribution require additional provenance.


In [4]:
contrasts = mp.contrast_statistics(cohorts, CHANNEL, OUT)
primary = contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.eq('clashscore')]
display(primary[['A','B','N_A','N_B','n_seeds_A','n_seeds_B','common_seeds',
                 'common_model_snapshots','only_A_snapshots','only_B_snapshots',
                 'paired_seed_correlation','primary_test']])
display(primary[['A','B','median_difference','seed_mean_difference','CI_low','CI_high',
                 'seed_rank_biserial','q_BH','independent_q_BH','common_seed_difference',
                 'common_seed_CI_low','common_seed_CI_high','common_seed_q_BH']].round(4))


,A,B,N_A,N_B,n_seeds_A,n_seeds_B,common_seeds,common_model_snapshots,only_A_snapshots,only_B_snapshots,paired_seed_correlation,primary_test
27,WT | vanilla,WT | masked v3,4591,4366,100,100,85,3488,1103,878,0.106477,Welch seed t
30,L403A | vanilla,L403A | masked v3,4403,4599,100,100,78,3284,1119,1315,0.065120,Welch seed t
33,F412L | vanilla,F412L | masked v3,4521,3643,100,100,55,1925,2596,1718,-0.254457,Welch seed t
36,WT | vanilla,L403A | vanilla,4591,4403,100,100,93,3973,618,430,0.072098,Welch seed t
39,WT | vanilla,F412L | vanilla,4591,4521,100,100,70,3011,1580,1510,0.050513,Welch seed t
42,L403A | vanilla,F412L | vanilla,4403,4521,100,100,77,3170,1233,1351,0.136270,Welch seed t
45,WT | masked v3,L403A | masked v3,4366,4599,100,100,100,4195,171,404,-0.057110,paired nominal seed t
48,WT | masked v3,F412L | masked v3,4366,3643,100,100,100,3431,935,212,-0.117386,paired nominal seed t
51,L403A | masked v3,F412L | masked v3,4599,3643,100,100,100,3561,1038,82,0.228923,paired nominal seed t


,A,B,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH,independent_q_BH,common_seed_difference,common_seed_CI_low,common_seed_CI_high,common_seed_q_BH
27,WT | vanilla,WT | masked v3,-1.83,-3.9463,-4.7927,-3.1029,-0.6176,0.0000,0.0000,-4.0930,-4.9960,-3.2161,0.0000
30,L403A | vanilla,L403A | masked v3,-1.60,-3.3664,-4.1577,-2.5820,-0.5770,0.0000,0.0000,-3.4192,-4.2550,-2.5623,0.0000
33,F412L | vanilla,F412L | masked v3,0.58,0.0137,-0.7243,0.7435,0.0062,0.9707,0.9707,0.0455,-1.0114,1.0780,0.9320
36,WT | vanilla,L403A | vanilla,0.92,0.3295,-0.6476,1.3206,0.0562,0.6038,0.6038,0.3748,-0.5960,1.3814,0.5737
39,WT | vanilla,F412L | vanilla,1.25,-0.3354,-1.2268,0.5183,-0.0378,0.5688,0.5688,-0.3770,-1.4725,0.6552,0.5737
42,L403A | vanilla,F412L | vanilla,0.33,-0.6648,-1.5239,0.2030,-0.1156,0.2186,0.2186,-0.8165,-1.7179,0.1397,0.1487
45,WT | masked v3,L403A | masked v3,1.15,0.9093,0.2544,1.5359,0.2542,0.0131,0.0104,0.9093,0.2544,1.5359,0.0141
48,WT | masked v3,F412L | masked v3,3.66,3.6246,2.9001,4.3723,0.6880,0.0000,0.0000,3.6246,2.9001,4.3723,0.0000
51,L403A | masked v3,F412L | masked v3,2.51,2.7152,2.1682,3.2705,0.6156,0.0000,0.0000,2.7152,2.1682,3.2705,0.0000


## Clashscore distributions and effect sizes
Violins retain every eligible individual snapshot with median and quartiles; no outliers are clipped. The paired raw/QC panels expose selection effects. The contrast figure shows seed-balanced differences and 95% bootstrap intervals; q-values are reported without significance stars. Colors, typography, thin outlines, light grids, and PDF/300-dpi PNG exports follow `shared.plotting` and existing conformational figures.

In [5]:
mp.plot_distributions(cohorts, CHANNEL, OUT)
plt.show()
mp.plot_effects(contrasts, CHANNEL, OUT)
plt.show()
display(Markdown(mp.interpretation(summary, contrasts, CHANNEL)))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_79004/2804364640.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_79004/2804364640.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Results on the repository QC subset

Model medians describe retained snapshots. Differences and confidence intervals below use equal AF-model weights within seed and seed-level inference.
- **WT | masked v3 versus WT | vanilla:** model medians 41.25 versus 43.08 (Δ -1.83); seed-balanced mean Δ -3.95, 95% CI [-4.79, -3.10]; seed rank-biserial effect -0.62; BH q=2.99e-15.
- **L403A | masked v3 versus L403A | vanilla:** model medians 42.40 versus 44.00 (Δ -1.60); seed-balanced mean Δ -3.37, 95% CI [-4.16, -2.58]; seed rank-biserial effect -0.58; BH q=1.12e-13.
- **F412L | masked v3 versus F412L | vanilla:** model medians 44.91 versus 44.33 (Δ +0.58); seed-balanced mean Δ +0.01, 95% CI [-0.72, 0.74]; seed rank-biserial effect +0.01; BH q=0.971.
- **L403A | vanilla versus WT | vanilla:** model medians 44.00 versus 43.08 (Δ +0.92); seed-balanced mean Δ +0.33, 95% CI [-0.65, 1.32]; seed rank-biserial effect +0.06; BH q=0.604.
- **F412L | vanilla versus WT | vanilla:** model medians 44.33 versus 43.08 (Δ +1.25); seed-balanced mean Δ -0.34, 95% CI [-1.23, 0.52]; seed rank-biserial effect -0.04; BH q=0.569.
- **F412L | vanilla versus L403A | vanilla:** model medians 44.33 versus 44.00 (Δ +0.33); seed-balanced mean Δ -0.66, 95% CI [-1.52, 0.20]; seed rank-biserial effect -0.12; BH q=0.219.
- **L403A | masked v3 versus WT | masked v3:** model medians 42.40 versus 41.25 (Δ +1.15); seed-balanced mean Δ +0.91, 95% CI [0.25, 1.54]; seed rank-biserial effect +0.25; BH q=0.0131.
- **F412L | masked v3 versus WT | masked v3:** model medians 44.91 versus 41.25 (Δ +3.66); seed-balanced mean Δ +3.62, 95% CI [2.90, 4.37]; seed rank-biserial effect +0.69; BH q=2.99e-15.
- **F412L | masked v3 versus L403A | masked v3:** model medians 44.91 versus 42.40 (Δ +2.51); seed-balanced mean Δ +2.72, 95% CI [2.17, 3.27]; seed rank-biserial effect +0.62; BH q=2.99e-15.

## Complementary steric diagnostics
These panels use the repository QC subset. Counts depend on structure size and atom handling; maximum overlap reports one extreme contact. Full numerical results and corrected comparisons for both metrics are retained alongside clashscore.

In [6]:
mp.plot_diagnostics(cohorts, CHANNEL, OUT)
plt.show()
display(summary[(summary.cohort=='repository_QC') & summary.metric.ne('clashscore')].round(3))
display(contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.ne('clashscore')][
    ['A','B','metric','median_difference','seed_mean_difference','CI_low','CI_high','seed_rank_biserial','q_BH']].round(4))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_79004/1691443703.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
55,repository_QC,F412L | masked v3,n_bad_clashes,3643,100,1732.000,1543.500,1908.000,364.500,1753.458,386.153,974.000,4569.000
56,repository_QC,F412L | masked v3,max_overlap,3643,100,1.628,1.596,1.654,0.058,1.622,0.048,1.438,1.763
58,repository_QC,F412L | vanilla,n_bad_clashes,4521,100,1710.000,1354.000,1974.000,620.000,1750.444,488.635,873.000,4180.000
59,repository_QC,F412L | vanilla,max_overlap,4521,100,1.629,1.597,1.655,0.058,1.622,0.049,1.426,1.787
61,repository_QC,L403A | masked v3,n_bad_clashes,4599,100,1634.000,1400.000,1832.000,432.000,1657.392,358.955,977.000,5999.000
62,repository_QC,L403A | masked v3,max_overlap,4599,100,1.630,1.603,1.657,0.054,1.628,0.042,1.460,1.778
64,repository_QC,L403A | vanilla,n_bad_clashes,4403,100,1695.000,1438.500,1975.500,537.000,1772.939,470.269,874.000,5172.000
65,repository_QC,L403A | vanilla,max_overlap,4403,100,1.627,1.599,1.653,0.054,1.624,0.045,1.441,1.780
67,repository_QC,WT | masked v3,n_bad_clashes,4366,100,1591.000,1405.000,1756.000,351.000,1619.731,356.116,946.000,9045.000
68,repository_QC,WT | masked v3,max_overlap,4366,100,1.623,1.594,1.647,0.053,1.618,0.044,1.451,1.745


,A,B,metric,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH
28,WT | vanilla,WT | masked v3,n_bad_clashes,-71.000,-150.7712,-183.1630,-118.4674,-0.6158,0.0000
29,WT | vanilla,WT | masked v3,max_overlap,-0.005,-0.0078,-0.0101,-0.0054,-0.4834,0.0000
31,L403A | vanilla,L403A | masked v3,n_bad_clashes,-61.000,-128.0801,-158.4352,-98.0839,-0.5740,0.0000
32,L403A | vanilla,L403A | masked v3,max_overlap,0.003,0.0033,0.0010,0.0055,0.2024,0.0114
34,F412L | vanilla,F412L | masked v3,n_bad_clashes,22.000,1.6699,-26.6587,29.6331,0.0132,0.9420
35,F412L | vanilla,F412L | masked v3,max_overlap,-0.001,-0.0004,-0.0031,0.0024,0.0036,0.8541
37,WT | vanilla,L403A | vanilla,n_bad_clashes,33.000,11.1166,-26.2760,49.1046,0.0504,0.6363
38,WT | vanilla,L403A | vanilla,max_overlap,-0.001,-0.0016,-0.0040,0.0006,-0.1040,0.2682
40,WT | vanilla,F412L | vanilla,n_bad_clashes,48.000,-12.9517,-47.0940,19.7362,-0.0390,0.5688
41,WT | vanilla,F412L | vanilla,max_overlap,0.001,-0.0028,-0.0052,-0.0004,-0.1862,0.0417


## Recycle, AF-model, seed, and rank sensitivity
The full snapshot trace includes final separately and all early-recycle tails. The final-only analysis does not substitute a last available snapshot for a missing final file. The adjusted regression controls AF-model and snapshot composition, with uncertainty clustered by seed; its observation-weighted estimand differs from the seed-balanced primary result.

In [7]:
mp.stratification(cohorts, CHANNEL, OUT)
plt.show()
adjusted = mp.adjusted_contrasts(cohorts, CHANNEL, OUT)
display(adjusted.round(4))
af_sensitivity = mp.af_model_sensitivity(cohorts, CHANNEL, OUT)
display(af_sensitivity.pivot(index=['A','B','mode'], columns='AF_model', values='seed_mean_difference').round(3))
display(contrasts[contrasts.metric.eq('clashscore')][
    ['cohort','A','B','seed_mean_difference','CI_low','CI_high','q_BH']].round(3))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_79004/2647556288.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,A,B,adjusted_difference,CI_low,CI_high,p,q_BH
0,WT | vanilla,WT | masked v3,-4.0009,-4.7401,-3.2617,0.0000,0.0000
1,L403A | vanilla,L403A | masked v3,-3.4753,-4.2037,-2.7470,0.0000,0.0000
2,F412L | vanilla,F412L | masked v3,0.1305,-0.5948,0.8559,0.7243,0.7243
3,WT | vanilla,L403A | vanilla,0.6080,-0.3329,1.5488,0.2053,0.2640
4,WT | vanilla,F412L | vanilla,-0.3770,-1.2572,0.5032,0.4012,0.4514
5,L403A | vanilla,F412L | vanilla,-0.9584,-1.8054,-0.1114,0.0266,0.0399
6,WT | masked v3,L403A | masked v3,1.2730,0.6409,1.9052,0.0001,0.0001
7,WT | masked v3,F412L | masked v3,3.8160,3.1378,4.4942,0.0000,0.0000
8,L403A | masked v3,F412L | masked v3,2.5618,2.0081,3.1156,0.0000,0.0000


AF_model                                                    1      2      3  \
A                 B                 mode                                      
F412L | vanilla   F412L | masked v3 leave_AF_model_out -0.130  0.352 -1.815   
                                    only_AF_model      -0.241 -1.860  7.268   
L403A | masked v3 F412L | masked v3 leave_AF_model_out  2.918  3.878  2.905   
                                    only_AF_model       1.207 -2.228  2.153   
L403A | vanilla   F412L | vanilla   leave_AF_model_out -1.808 -0.653 -0.569   
                                    only_AF_model       3.944 -0.473 -0.990   
                  L403A | masked v3 leave_AF_model_out -4.855 -4.179 -5.288   
                                    only_AF_model       2.496 -0.105  4.126   
WT | masked v3    F412L | masked v3 leave_AF_model_out  3.362  4.108  3.530   
                                    only_AF_model       4.202  1.503  3.950   
                  L403A | masked v3 leave_AF_model_out  0.444  0.230  0.625   
                                    only_AF_model       2.995  3.731  1.798   
WT | vanilla      F412L | vanilla   leave_AF_model_out -1.602 -0.695  0.164   
                                    only_AF_model       4.751  1.166 -2.247   
                  L403A | vanilla   leave_AF_model_out  0.205 -0.041  0.733   
                                    only_AF_model       0.807  1.639 -1.257   
                  WT | masked v3    leave_AF_model_out -5.094 -4.450 -5.180   
                                    only_AF_model       0.308 -2.198  1.072   

AF_model                                                     4       5  
A                 B                 mode                                
F412L | vanilla   F412L | masked v3 leave_AF_model_out  -0.543   2.204  
                                    only_AF_model        1.924  -8.113  
L403A | masked v3 F412L | masked v3 leave_AF_model_out   1.253   2.623  
                                    only_AF_model        8.820   3.749  
L403A | vanilla   F412L | vanilla   leave_AF_model_out   0.742  -1.036  
                                    only_AF_model       -6.560   0.894  
                  L403A | masked v3 leave_AF_model_out  -1.054  -1.456  
                                    only_AF_model      -13.456 -10.969  
WT | masked v3    F412L | masked v3 leave_AF_model_out   2.722   4.401  
                                    only_AF_model        8.946   1.030  
                  L403A | masked v3 leave_AF_model_out   1.470   1.778  
                                    only_AF_model        0.127  -2.719  
WT | vanilla      F412L | vanilla   leave_AF_model_out   0.725  -0.269  
                                    only_AF_model       -4.612  -0.398  
                  L403A | vanilla   leave_AF_model_out  -0.017   0.768  
                                    only_AF_model        1.948  -1.292  
                  WT | masked v3    leave_AF_model_out  -2.541  -2.466  
                                    only_AF_model      -11.635  -9.542

,cohort,A,B,seed_mean_difference,CI_low,CI_high,q_BH
0,all_recycles,WT | vanilla,WT | masked v3,-3.591,-4.549,-2.604,0.000
3,all_recycles,L403A | vanilla,L403A | masked v3,-4.769,-5.577,-3.995,0.000
6,all_recycles,F412L | vanilla,F412L | masked v3,2.049,1.246,2.827,0.000
9,all_recycles,WT | vanilla,L403A | vanilla,1.129,0.171,2.113,0.033
12,all_recycles,WT | vanilla,F412L | vanilla,-0.172,-1.067,0.722,0.766
15,all_recycles,L403A | vanilla,F412L | vanilla,-1.302,-2.163,-0.399,0.008
18,all_recycles,WT | masked v3,L403A | masked v3,-0.049,-0.877,0.779,0.909
21,all_recycles,WT | masked v3,F412L | masked v3,5.467,4.535,6.371,0.000
24,all_recycles,L403A | masked v3,F412L | masked v3,5.516,4.933,6.112,0.000
27,repository_QC,WT | vanilla,WT | masked v3,-3.946,-4.793,-3.103,0.000


## S6 conformational association: join first, test second

The existing chain-resolved table contains all six ensembles. Join on **ensemble + exact PDB basename**,
require one record per canonical chain, and cross-check seed, AF model, and snapshot. No rank-based
matching or fuzzy filename normalization is permitted. The audit covers all recorded models; the
primary association uses the exact final structural-QC subset.

Two complementary existing-coordinate summaries are examined:

- Median canonical-chain F412-position Cα displacement from 8SD3 (Å), an **unsigned** magnitude.
  For F412L, the existing column refers to the homologous L412 Cα anchor.
- A directional projection of the four canonical-chain **whole-S6 rotations** onto the experimental
  8SD3→8SDA rotation vector: `dot(model − 8SD3, 8SDA − 8SD3) / norm(8SDA − 8SD3)^2`.
  Angular differences wrap to [−180°,180°). Zero is the WT rotation reference, one the mutant-reference
  projection. This is a derived summary of the existing coordinate, not a state classifier, a complete
  3-D conformation match, or a claim that projection one reproduces all four chains. Canonical subunit
  mapping is inherited from the validated analysis, not inferred from MolProbity.

Spearman correlation uses seed summaries (median per trajectory, equal AF-model mean within seed),
with 1,000 seed bootstrap draws. Descriptive model-level rho is also exported. Regressions adjust for
AF model/snapshot and cluster uncertainty by seed; slope × coordinate IQR expresses a local magnitude.
Direct interaction tests compare slopes between conditions rather than comparing significance labels.
The 12 association tests per method/cohort and the 18 interaction tests form separate BH families.
Raw-recycle, final-only, and r10 within the same structural-QC population are sensitivity analyses. Correlation cannot establish that
steric strain causes or limits S6 movement; early-recycle quality and survivor selection can influence it.


In [8]:
joined, associations, interactions = mp.kv21_conformation(cohorts, OUT)
display(pd.read_csv(OUT/'tables/conformation_join_audit.csv'))
display(pd.read_csv(OUT/'tables/S6_rotation_reference.csv'))
display(associations[associations.cohort.eq('repository_QC')].round(4))
display(interactions.round(4))
mp.plot_conformation(joined, associations, OUT)
plt.show()
mp.plot_conformation_adjusted(joined, associations, OUT)
plt.show()
display(associations[associations.cohort.eq('r10_repository_QC')].round(4))


/Users/ahernandezgonzalez/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DataUnavailable: kv21/dataExtra/kv21_l403a_conformational_metrics_chain_resolved.csv is not in the working tree and could not be fetched from adrishgz/vgic-mutant-structural-ensembles as data/Kv2.1/extra/kv21_l403a_conformational_metrics_chain_resolved.csv: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

## Focused interpretation
The following text is generated from the executed result tables so it stays synchronized with reruns.

In [9]:
display(Markdown(mp.focused_interpretation(summary, contrasts, CHANNEL, OUT)))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ahernandezgonzalez/Repositories/vgci_mutants/kv21/dataExtra/molprobity/tables/conformation_associations.csv'

## Reproducibility record
All generated outputs are confined to this channel’s `dataExtra/molprobity/{tables,figures}`. Existing figures and analyses are not overwritten. Input hashes, missing-model lists, full descriptive/contrast tables, pairing diagnostics and sensitivity results accompany the figures.

In [10]:
record = {'channel': CHANNEL, 'total_downloaded_csvs': total_csv_files,
          'channel_csvs': len(inventory), 'rows': len(raw), 'python': platform.python_version(),
          'bootstrap_replicates': mp.BOOTSTRAPS, 'random_seed': mp.RANDOM_SEED,
          'packages': {p: importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','matplotlib','seaborn']}}
(OUT/'tables/run_metadata.json').write_text(json.dumps(record, indent=2)+chr(10))
print('Completed:', OUT.relative_to(ROOT))


Completed: kv21/dataExtra/molprobity
